In [1]:
# Colab already has Java and (usually) PySpark. Install only if missing.
import importlib.util, subprocess, sys
if importlib.util.find_spec('pyspark') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark', 'pyarrow'], check=True)
import pyspark; print('pyspark', pyspark.__version__)

pyspark 4.0.4


> **Standalone notebook.** Colab starts a fresh session for each notebook, so the first cells below download and rebuild whatever earlier stages produced (the raw data lake from Stage 1). This takes a few minutes the first time in each session.

In [2]:
# ===== Self-contained setup: rebuilds the data this notebook needs in a fresh Colab session =====
import os, math, time, requests
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

YEAR, MONTHS = 2024, [1, 2, 3]
BASE = Path('..') if Path('../data').exists() else Path('.')
RAW, LAKE = BASE / 'data' / 'raw', BASE / 'data' / 'lake'
for _p in (RAW, LAKE, BASE / 'results', BASE / 'models'):
    _p.mkdir(parents=True, exist_ok=True)
URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{y}-{m:02d}.parquet'
ZONES_URL = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

def get_spark(name='nyc-taxi'):
    return (SparkSession.builder.appName(name).master('local[*]')
            .config('spark.driver.memory', '4g').config('spark.sql.shuffle.partitions', '16').getOrCreate())

def _download(url, dest):
    if dest.exists():
        return
    r = requests.get(url, stream=True, timeout=120); r.raise_for_status()
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    print('downloaded', dest.name, round(dest.stat().st_size / 1e6, 1), 'MB')

def ensure_raw_zone():
    """Download TLC files and write the partitioned Parquet raw zone (same as Stage 1)."""
    if (LAKE / 'raw_zone' / 'yellow_trips').exists() and (LAKE / 'raw_zone' / 'taxi_zones').exists():
        print('raw zone ready'); return
    spark = get_spark(); files = []
    for m in MONTHS:
        dest = RAW / f'yellow_tripdata_{YEAR}-{m:02d}.parquet'
        _download(URL.format(y=YEAR, m=m), dest); files.append(str(dest))
    _download(ZONES_URL, RAW / 'taxi_zone_lookup.csv')
    df = spark.read.parquet(*files).withColumn('pickup_month', F.date_format('tpep_pickup_datetime', 'yyyy-MM'))
    df.write.mode('overwrite').partitionBy('pickup_month').parquet(str(LAKE / 'raw_zone' / 'yellow_trips'))
    zones = spark.read.csv(str(RAW / 'taxi_zone_lookup.csv'), header=True, inferSchema=True)
    zones.write.mode('overwrite').parquet(str(LAKE / 'raw_zone' / 'taxi_zones'))
    print('raw zone written')

def ensure_curated_zone():
    """Clean, engineer features and split 70/15/15 (same rules as Stage 2)."""
    cz = LAKE / 'curated_zone'
    if (cz / 'test').exists():
        print('curated zone ready'); return
    ensure_raw_zone(); spark = get_spark()
    cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance',
            'RatecodeID', 'PULocationID', 'DOLocationID', 'fare_amount', 'pickup_month']
    df = spark.read.parquet(str(LAKE / 'raw_zone' / 'yellow_trips')).select(*cols).dropna().dropDuplicates()
    df = (df.withColumn('duration_min', (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60)
            .withColumn('speed_mph', F.col('trip_distance') / (F.col('duration_min') / 60)))
    valid_months = [f'{YEAR}-{m:02d}' for m in MONTHS]
    df = df.filter(F.col('pickup_month').isin(valid_months) & F.col('fare_amount').between(3, 200) &
                   F.col('trip_distance').between(0.1, 100) & F.col('passenger_count').between(1, 6) &
                   F.col('RatecodeID').between(1, 6) & F.col('PULocationID').between(1, 263) &
                   F.col('DOLocationID').between(1, 263) & F.col('duration_min').between(1, 180) &
                   (F.col('speed_mph') < 80))
    df = (df.withColumn('hour', F.hour('tpep_pickup_datetime'))
            .withColumn('dow', (F.dayofweek('tpep_pickup_datetime') + 5) % 7)
            .withColumn('is_weekend', (F.col('dow') >= 5).cast('int'))
            .withColumn('hour_sin', F.sin(2 * math.pi * F.col('hour') / 24))
            .withColumn('hour_cos', F.cos(2 * math.pi * F.col('hour') / 24)))
    for c in ['passenger_count', 'RatecodeID', 'PULocationID', 'DOLocationID']:
        df = df.withColumn(c, F.col(c).cast('int'))
    df = (df.withColumn('trip_distance', F.col('trip_distance').cast('double'))
            .withColumn('fare_amount', F.col('fare_amount').cast('double'))
            .select('trip_distance', 'passenger_count', 'hour', 'dow', 'is_weekend', 'hour_sin', 'hour_cos',
                    'PULocationID', 'DOLocationID', 'RatecodeID', 'fare_amount', 'pickup_month'))
    df.write.mode('overwrite').parquet(str(cz / 'all'))      # materialise once -> reproducible split
    df = spark.read.parquet(str(cz / 'all')).withColumn('_r', F.rand(42))
    df.filter(F.col('_r') < 0.70).drop('_r').write.mode('overwrite').parquet(str(cz / 'train'))
    df.filter((F.col('_r') >= 0.70) & (F.col('_r') < 0.85)).drop('_r').write.mode('overwrite').parquet(str(cz / 'val'))
    df.filter(F.col('_r') >= 0.85).drop('_r').write.mode('overwrite').parquet(str(cz / 'test'))
    print('curated zone written')

def ensure_model_zone(frac=0.20):
    """Export the pandas-friendly sample used for neural-network training (same as Stage 3)."""
    mz, cz = LAKE / 'model_zone', LAKE / 'curated_zone'
    if (mz / 'test').exists():
        print('model zone ready'); return
    ensure_curated_zone(); spark = get_spark()
    for name in ['train', 'val', 'test']:
        s = spark.read.parquet(str(cz / name)).drop('pickup_month').sample(False, frac, seed=42)
        s.write.mode('overwrite').parquet(str(mz / name))
    print('model zone written')

ensure_raw_zone()

downloaded yellow_tripdata_2024-01.parquet 50.0 MB
downloaded yellow_tripdata_2024-02.parquet 50.3 MB
downloaded yellow_tripdata_2024-03.parquet 60.1 MB
downloaded taxi_zone_lookup.csv 0.0 MB
raw zone written


# Stage 2: Data Preparation (PySpark)
Cleaning, missing values, duplicates, outliers, feature engineering and the train/validation/test split.
The result is written to the **curated zone** of the data lake.

**Prediction task:** predict `fare_amount` (regression). Features are available at booking time (distance estimate, time, zones, passengers, rate code).
Leakage control: tip, tolls, total_amount, duration and payment type are excluded because they are not known before the trip.

In [3]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

BASE = Path('..') if Path('../data').exists() else Path('.')
LAKE = BASE / 'data' / 'lake'
YEAR, MONTHS = 2024, [1, 2, 3]   # must match Stage 1

spark = (SparkSession.builder.appName('nyc-taxi-stage2').master('local[*]')
         .config('spark.driver.memory', '4g').config('spark.sql.shuffle.partitions', '16').getOrCreate())

## 1. Load raw zone and keep relevant columns

In [4]:
raw = spark.read.parquet(str(LAKE / 'raw_zone' / 'yellow_trips'))
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance',
        'RatecodeID', 'PULocationID', 'DOLocationID', 'fare_amount', 'pickup_month']
df = raw.select(*cols)
report = {'raw_rows': df.count()}
print(report)

{'raw_rows': 9554778}


## 2. Missing values
Rows with nulls in any modelling column are dropped. Imputing fare, distance or location would invent information about the target-related fields, and the null share is small (check Stage 1 output).

In [5]:
df = df.dropna()
report['after_dropna'] = df.count()
print(report)

{'raw_rows': 9554778, 'after_dropna': 8802816}


## 3. Duplicates

In [6]:
df = df.dropDuplicates()
report['after_duplicates'] = df.count()
print(report)

{'raw_rows': 9554778, 'after_dropna': 8802816, 'after_duplicates': 8802483}


## 4. Invalid values and outliers (rule-based)
Rules (each is a documented decision, justify them in the report):
- pickup month must belong to the downloaded months (the raw files contain a few mis-dated records)
- `fare_amount` 3 to 200 USD (3 is roughly the minimum metered fare; negatives are refunds/errors)
- `trip_distance` 0.1 to 100 miles
- `passenger_count` 1 to 6
- `RatecodeID` 1 to 6 (99 is an unknown code)
- zones 1 to 263 (264/265 are the 'unknown' zones)
- duration 1 to 180 minutes and average speed under 80 mph (removes impossible trips)

In [7]:
valid_months = [f'{YEAR}-{m:02d}' for m in MONTHS]
df = (df.withColumn('duration_min', (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60)
        .withColumn('speed_mph', F.col('trip_distance') / (F.col('duration_min') / 60)))

df = df.filter(
    F.col('pickup_month').isin(valid_months) &
    F.col('fare_amount').between(3, 200) &
    F.col('trip_distance').between(0.1, 100) &
    F.col('passenger_count').between(1, 6) &
    F.col('RatecodeID').between(1, 6) &
    F.col('PULocationID').between(1, 263) & F.col('DOLocationID').between(1, 263) &
    F.col('duration_min').between(1, 180) &
    (F.col('speed_mph') < 80))
report['after_validity_filters'] = df.count()
print(report)

{'raw_rows': 9554778, 'after_dropna': 8802816, 'after_duplicates': 8802483, 'after_validity_filters': 8273082}


## 5. Feature engineering and encoding
- `hour`, `dow` (0=Mon), `is_weekend` from the pickup timestamp
- cyclical encoding `hour_sin`, `hour_cos` so that 23:00 and 00:00 are close
- zone IDs and rate code stay as integer IDs; the neural network learns **embeddings** for them in Stage 5 (instead of 260+ one-hot columns)
- scaling of numeric features is fitted on the training split only (Stage 4) to avoid leakage

In [8]:
import math
df = (df.withColumn('hour', F.hour('tpep_pickup_datetime'))
        .withColumn('dow', (F.dayofweek('tpep_pickup_datetime') + 5) % 7)   # Spark: Sunday=1 -> convert to Monday=0
        .withColumn('is_weekend', (F.col('dow') >= 5).cast('int'))
        .withColumn('hour_sin', F.sin(2 * math.pi * F.col('hour') / 24))
        .withColumn('hour_cos', F.cos(2 * math.pi * F.col('hour') / 24))
        .withColumn('passenger_count', F.col('passenger_count').cast('int'))
        .withColumn('RatecodeID', F.col('RatecodeID').cast('int'))
        .withColumn('PULocationID', F.col('PULocationID').cast('int'))
        .withColumn('DOLocationID', F.col('DOLocationID').cast('int'))
        .withColumn('trip_distance', F.col('trip_distance').cast('double'))
        .withColumn('fare_amount', F.col('fare_amount').cast('double')))

final_cols = ['trip_distance', 'passenger_count', 'hour', 'dow', 'is_weekend', 'hour_sin', 'hour_cos',
              'PULocationID', 'DOLocationID', 'RatecodeID', 'fare_amount', 'pickup_month']
df = df.select(*final_cols)
df.show(5)

+-------------+---------------+----+---+----------+--------+--------+------------+------------+----------+-----------+------------+
|trip_distance|passenger_count|hour|dow|is_weekend|hour_sin|hour_cos|PULocationID|DOLocationID|RatecodeID|fare_amount|pickup_month|
+-------------+---------------+----+---+----------+--------+--------+------------+------------+----------+-----------+------------+
|          2.4|              1|   0|  4|         0|     0.0|     1.0|         237|         234|         1|       12.8|     2024-03|
|         1.24|              1|   0|  4|         0|     0.0|     1.0|         161|         170|         1|        9.3|     2024-03|
|         1.92|              1|   0|  4|         0|     0.0|     1.0|         162|          90|         1|       12.8|     2024-03|
|         0.67|              1|   0|  4|         0|     0.0|     1.0|         231|         211|         1|        5.8|     2024-03|
|          3.0|              1|   0|  4|         0|     0.0|     1.0|       

## 6. Train / validation / test split (70 / 15 / 15)
Random split with a fixed seed. Test data is touched only once, in Stage 6.

In [9]:
cz = str(LAKE / 'curated_zone')
# Materialise the cleaned data once, then split the re-read files: this keeps the random split reproducible
df.write.mode('overwrite').parquet(cz + '/all')
df = spark.read.parquet(cz + '/all').withColumn('_r', F.rand(42))
train = df.filter(F.col('_r') < 0.70).drop('_r')
val   = df.filter((F.col('_r') >= 0.70) & (F.col('_r') < 0.85)).drop('_r')
test  = df.filter(F.col('_r') >= 0.85).drop('_r')

cz = str(LAKE / 'curated_zone')
train.write.mode('overwrite').parquet(cz + '/train')
val.write.mode('overwrite').parquet(cz + '/val')
test.write.mode('overwrite').parquet(cz + '/test')

for name in ['train', 'val', 'test']:
    report[name + '_rows'] = spark.read.parquet(f'{cz}/{name}').count()
print(report)

{'raw_rows': 9554778, 'after_dropna': 8802816, 'after_duplicates': 8802483, 'after_validity_filters': 8273082, 'train_rows': 5791911, 'val_rows': 1240671, 'test_rows': 1240500}


In [10]:
import json, pandas as pd
(BASE / 'results').mkdir(exist_ok=True)
(BASE / 'results' / 'cleaning_report.json').write_text(json.dumps(report, indent=2))
pd.Series(report, name='rows').to_frame()

,rows
raw_rows,9554778
after_dropna,8802816
after_duplicates,8802483
after_validity_filters,8273082
train_rows,5791911
val_rows,1240671
test_rows,1240500


## Notes for the report
- Copy the cleaning report table (rows removed at each step and percentage).
- Explain every threshold above in one sentence.
- Next: Stage 3, Big Data processing and analysis with Spark.

In [11]:
# Optional: download this stage's results (tables/figures) because the Colab session is temporary
try:
    import shutil
    from google.colab import files
    shutil.make_archive('/content/results_stage2', 'zip', str(BASE / 'results'))
    files.download('/content/results_stage2.zip')
except Exception as e:
    print('download skipped:', e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>